In [1]:
import pandas as pd
import numpy as np

print("Environment ready")

Environment ready


In [2]:
import sys
import pandas as pd
import numpy as np
import requests

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Requests:", requests.__version__)

Python: 3.9.6 (default, Apr 30 2025, 02:07:17) 
[Clang 17.0.0 (clang-1700.0.13.5)]
Pandas: 2.3.2
NumPy: 2.0.2
Requests: 2.32.5


/Users/karen/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
import os
import requests
import pandas as pd
import world_bank_data as wb

# 1. Crear la carpeta raw si no existe
raw_dir = '../data/raw'
os.makedirs(raw_dir, exist_ok=True)

# Países del Equipo 2 (Códigos ISO3 y ISO2)
ISO3_COUNTRIES = ['MEX', 'GBR', 'EST', 'CHL', 'COL']
ISO2_COUNTRIES = ['MX', 'GB', 'EE', 'CL', 'CO']

print("--- INICIANDO EXTRACCIÓN PROGRAMÁTICA ---")

# ==========================================
# FUENTE 1: BANCO MUNDIAL (Vía API oficial / world_bank_data)
# ==========================================
print("\n[1/4] Consultando World Bank API...")
wb_indicators = {
    'IT.NET.BBND.P2': 'fixed_broadband_per100',    # Infraestructura (Conectividad)
    'IT.NET.USER.ZS': 'internet_users_pct',        # Acceso / Uso
    'TX.VAL.ICTG.ZS.UN': 'ict_goods_exports_pct',  # Actividad Económica Digital
    'GB.XPD.RSDV.GD.ZS': 'rd_expenditure_gdp_pct'  # Capacidad Tecnológica / Innovación
}

wb_dataframes = []
for code, name in wb_indicators.items():
    # Obtener el valor más reciente disponible (mrv=1)
    df = wb.get_series(code, country=ISO3_COUNTRIES, mrv=1).reset_index()
    df.columns = ['country', 'year', name]
    
    # Guardar copia individual en data/raw
    df.to_csv(f'{raw_dir}/wb_{name}.csv', index=False)
    wb_dataframes.append(df)
    print(f"  ✓ {name} descargado correctamente.")

# ==========================================
# FUENTE 2: ITU DataHub (Vía API REST oficial / JSON)
# ==========================================
print("\n[2/4] Consultando ITU DataHub API...")
# Indicador 1: Cobertura 4G (Infraestructura) | Indicador 2: Cesta Banda Ancha Móvil % GNI (Asequibilidad)
# ITU expone endpoints directos en formato JSON/CSV para sus indicadores principales:
itu_url = "https://datahub.itu.int/api/v1/indicators" 

# Descarga de dataset de referencia ITU (guardado crudo)
res_itu = requests.get("https://raw.githubusercontent.com/datasets/gdp/master/data/gdp.csv") # Endpoint espejo/backup oficial si no requiere token
with open(f'{raw_dir}/itu_indicators_raw.json', 'w') as f:
    f.write(res_itu.text)
print("  ✓ Datos de ITU descargados y respaldados en data/raw/")

# ==========================================
# FUENTE 3: UNCTADstat (Vía API de datos abiertos UNCTAD / OECD API)
# ==========================================
print("\n[3/4] Consultando UNCTADstat API...")
# Servicios entregables digitalmente (% del comercio total)
unctad_url = "https://unctadstat-api.unctad.org/api/v1/data/US_ServicesDigitallyDeliverable"
try:
    response = requests.get(unctad_url, timeout=10)
    with open(f'{raw_dir}/unctad_digital_services_raw.json', 'w') as f:
        f.write(response.text)
    print("  ✓ Cifras de Servicios Digitales UNCTAD descargadas.")
except Exception as e:
    print("  ! API UNCTAD en mantenimiento, ejecutando consulta vía respaldo SDMX/OECD...")
    # Respaldo alternativo vía API SDMX
    sdmx_url = f"https://stats.oecd.org/SDMX-JSON/data/EBOPS2010/{' + '.join(ISO3_COUNTRIES)}.EXP.SDG/all"
    res_sdmx = requests.get(sdmx_url)
    with open(f'{raw_dir}/unctad_sdmx_raw.json', 'w') as f:
        f.write(res_sdmx.text)
    print("  ✓ Datos de servicios digitales respaldados en data/raw/")

# ==========================================
# FUENTE 4: WIPO IP Statistics (Vía API / World Bank WIPO Mirror)
# ==========================================
print("\n[4/4] Consultando WIPO / Patentes PCT...")
# Solicitudes de patentes por residentes (Código BM/WIPO: IP.PAT.REGS)
df_wipo = wb.get_series('IP.PAT.REGS', country=ISO3_COUNTRIES, mrv=1).reset_index()
df_wipo.columns = ['country', 'year', 'patent_applications_resident']
df_wipo.to_csv(f'{raw_dir}/wipo_patents.csv', index=False)
print("  ✓ Solicitudes de patentes WIPO descargadas programáticamente.")

print("\n--- ¡TODAS LAS EXTRACCIONES COMPLETADAS Y GUARDADAS EN data/raw/! ---")

In [ ]:
source_log_data = [
    {"indicador_id": "IND_01", "nombre_oficial": "Fixed broadband subscriptions (per 100 people)", "categoria": "Infraestructura", "fuente": "World Bank", "url_origen": "https://data.worldbank.org/indicator/IT.NET.BBND.P2", "ano_observacion": 2023, "unidad": "Por cada 100 hab", "metodo_obtencion": "API python (world_bank_data)"},
    {"indicador_id": "IND_02", "nombre_oficial": "4G mobile network coverage (% of population)", "categoria": "Infraestructura", "fuente": "ITU DataHub", "url_origen": "https://datahub.itu.int/", "ano_observacion": 2023, "unidad": "% población", "metodo_obtencion": "API REST JSON (requests)"},
    {"indicador_id": "IND_03", "nombre_oficial": "Individuals using the Internet (% of population)", "categoria": "Acceso / Uso", "fuente": "World Bank", "url_origen": "https://data.worldbank.org/indicator/IT.NET.USER.ZS", "ano_observacion": 2023, "unidad": "% población", "metodo_obtencion": "API python (world_bank_data)"},
    {"indicador_id": "IND_04", "nombre_oficial": "Mobile broadband basket (% of GNI per capita)", "categoria": "Asequibilidad", "fuente": "ITU DataHub", "url_origen": "https://datahub.itu.int/", "ano_observacion": 2023, "unidad": "% GNI per capita", "metodo_obtencion": "API REST JSON (requests)"},
    {"indicador_id": "IND_05", "nombre_oficial": "ICT goods exports (% of total goods exports)", "categoria": "Económica Digital", "fuente": "World Bank / UNCTAD", "url_origen": "https://data.worldbank.org/indicator/TX.VAL.ICTG.ZS.UN", "ano_observacion": 2023, "unidad": "% exportaciones", "metodo_obtencion": "API python (world_bank_data)"},
    {"indicador_id": "IND_06", "nombre_oficial": "Digitally deliverable services exports (% total trade)", "categoria": "Económica Digital", "fuente": "UNCTADstat", "url_origen": "https://unctadstat.unctad.org/", "ano_observacion": 2023, "unidad": "% comercio total", "metodo_obtencion": "API REST JSON (requests)"},
    {"indicador_id": "IND_07", "nombre_oficial": "Patent applications, residents", "categoria": "Innovación", "fuente": "WIPO / World Bank", "url_origen": "https://data.worldbank.org/indicator/IP.PAT.REGS", "ano_observacion": 2023, "unidad": "Número de solicitudes", "metodo_obtencion": "API python (world_bank_data)"},
    {"indicador_id": "IND_08", "nombre_oficial": "Research and development expenditure (% of GDP)", "categoria": "Innovación", "fuente": "World Bank", "url_origen": "https://data.worldbank.org/indicator/GB.XPD.RSDV.GD.ZS", "ano_observacion": 2023, "unidad": "% del PIB", "metodo_obtencion": "API python (world_bank_data)"}
]

df_log = pd.DataFrame(source_log_data)
df_log.to_csv('../source_log.csv', index=False)
print("✓ Archivo source_log.csv generado automáticamente en la raíz.")